In [1]:
from pathlib import Path
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

SFT_MODEL_DIR = Path("../outputs/llm/gpt2-sales-sft")
RLHF_MODEL_DIR = Path("../outputs/llm/gpt2-sales-rlhf")

device = "mps" if torch.backends.mps.is_available() else "cpu"

/Users/imvenu/turing-ai-project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = GPT2TokenizerFast.from_pretrained(SFT_MODEL_DIR)
tokenizer.pad_token = tokenizer.eos_token

ref_model = GPT2LMHeadModel.from_pretrained(SFT_MODEL_DIR).to(device)
policy_model = GPT2LMHeadModel.from_pretrained(SFT_MODEL_DIR).to(device)

ref_model.eval()
policy_model.train()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
from datasets import Dataset
import random

# Load your SFT dataset
sft_df = pd.read_csv("../data/sft/sales_llm_instructions.csv")

preference_rows = []

for _, row in sft_df.sample(300, random_state=42).iterrows():
    prompt = row["instruction"]

    # Good answer (from SFT)
    chosen = row["output"]

    # Bad answer (intentionally weaker / generic)
    rejected = "Sales patterns vary and require monitoring."

    preference_rows.append({
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected
    })

pref_df = pd.DataFrame(preference_rows)

# Convert to HF Dataset
full_ds = Dataset.from_pandas(pref_df)

# Train / eval split
split = full_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds = split["test"]

print("✅ Preference dataset ready")
print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))

NameError: name 'pd' is not defined

In [4]:
from trl import DPOConfig

dpo_config = DPOConfig(
    output_dir=str(RLHF_MODEL_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    beta=0.1,
    max_length=256,
    max_prompt_length=128,
    remove_unused_columns=False,
)

In [4]:
from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

NameError: name 'train_ds' is not defined

In [5]:
dpo_trainer.train()

NameError: name 'dpo_trainer' is not defined

In [5]:
RLHF_MODEL_DIR.mkdir(parents=True, exist_ok=True)

dpo_trainer.save_model(str(RLHF_MODEL_DIR))
tokenizer.save_pretrained(str(RLHF_MODEL_DIR))

print("Saved RLHF model to:", RLHF_MODEL_DIR)

NameError: name 'dpo_trainer' is not defined

In [6]:
print("RLHF exists:", RLHF_MODEL_DIR.exists())
print("Files:")
for f in RLHF_MODEL_DIR.iterdir():
    print(" -", f.name)

RLHF exists: True
Files:
